# 03 最適化アルゴリズム

| アルゴリズム | 方法 | 評価回数 | 最適性 |
| --- | --- | --- | --- |
| 全探索 | 刻みごとの配分をすべて列挙して評価 | $C(M+3, 3)$ | 離散化した探索空間の中で最適 |
| Greedy 法 | 予算を 1 単位ずつ、改善が最大の分野へ | $M \times 4$ | 保証なし |
| 局所探索 | 1 単位を別の分野へ移す近傍のうち、最も良いものへ移る | 反復 × 近傍（最大 12） | 保証なし（局所最適） |

3 つとも**同じ目的関数・同じ経済シミュレーション・同じ探索空間**（RQ4）。計算：`optimization.py`

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 1. 目的関数と予算制約

- 予算を $M = 1/\text{刻み}$ 個の**整数単位**に分け、配分を $(u_1, u_2, u_3, u_4)$（合計 $M$）で表す → 配分率の合計は**ちょうど 1**（浮動小数点の誤差なし）
- 目的関数：`final_gdp`・`cumulative_gdp`・`final_living_standard`・`average_living_standard`・`weighted`
- 同点は「列挙の順で最初」（教科書の `max_of` + `seq_search`）→ 結果は決定的

In [ ]:
OBJECTIVE = "final_gdp"           # TODO：目的関数を変更する
print(f"刻み {STEP:.0%} → M = {M}、候補数 C(M+3, 3) = {op.count_allocations(M)}")
print("列挙の最初の 5 個:", op.enumerate_allocations(M)[:5])
res = op.compare_algorithms(PARAMS, OBJECTIVE, STEP)
rows = [{"name": r["algorithm"], "objective": r["objective"], "gap": r["gap"]["absolute"],
         "runtime_ms": r["runtime_s"] * 1e3, "evaluations": r["evaluations"], "iterations": r["iterations"],
         **r["shares"]} for r in res.values()]
table(rows, [("name", "algorithm", "{}"), ("objective", "objective", "{:.4f}"), ("gap", "gap vs exhaustive", "{:.4f}"),
             ("runtime_ms", "runtime (ms)", "{:.1f}"), ("evaluations", "evaluations", "{}"),
             ("iterations", "iterations", "{}")] + alloc_cols())

## 2. アルゴリズム別の目的関数値・実行時間・評価候補数

実行時間は**実行環境で変わる実測値**です。評価候補数（シミュレーションの回数）は環境によりません。

In [ ]:
#@title 目的関数値・実行時間・評価候補数
show(gc.plot_algorithm_metric(res, "objective", f"Objective ({op.OBJECTIVE_LABELS[OBJECTIVE]}) by algorithm (model)",
                              "objective", fmt="{:.2f}"), "03_objective")
show(gc.plot_algorithm_metric({k: dict(r, runtime_ms=r["runtime_s"] * 1e3) for k, r in res.items()}, "runtime_ms",
                              "Runtime by algorithm (measured on this machine)", "runtime (ms)", fmt="{:.1f}"),
     "03_runtime")
show(gc.plot_algorithm_metric(res, "evaluations", "Number of evaluated allocations (simulations)",
                              "evaluations", fmt="{:d}"), "03_evaluations")

## 3. 最適配分の比較

In [ ]:
#@title 最適配分と探索の途中経過
show(gc.plot_allocation_bars([(gc.ALGO_LABELS[k], r["shares"]) for k, r in res.items()],
                             f"Allocation found by each algorithm ({op.OBJECTIVE_LABELS[OBJECTIVE]})"), "03_allocations")
show(gc.plot_search_path(res["greedy"], "Greedy: objective after each added budget unit"), "03_greedy_path")
show(gc.plot_search_path(res["local_search"], "Local search: objective at each move"), "03_local_path")

## 4. 計算量：刻みを変える

候補数は $C(M+3, 3)$ で、刻みを半分にすると約 8 倍になります（$M^3$ に比例）。Greedy 法は $4M$ 回です。

In [ ]:
rows = []
for step in [0.25, 0.2, 0.1, 0.05]:                # TODO：配分刻みを変更する
    r = op.compare_algorithms(PARAMS, OBJECTIVE, step)
    for k, x in r.items():
        rows.append({"step": f"{step:.0%}", "name": x["algorithm"], "candidates": op.count_allocations(al.units_count(step)),
                     "evaluations": x["evaluations"], "runtime_ms": x["runtime_s"] * 1e3, "objective": x["objective"],
                     "gap": x["gap"]["absolute"]})
table(rows, [("step", "step", "{}"), ("name", "algorithm", "{}"), ("candidates", "candidates", "{}"),
             ("evaluations", "evaluations", "{}"), ("runtime_ms", "runtime (ms)", "{:.1f}"),
             ("objective", "objective", "{:.4f}"), ("gap", "gap", "{:.4f}")])

## 5. 学生向け TODO：初期配分・条件を変える

局所探索は**初期配分**によって結果が変わることがあります。Greedy 法・局所探索が全探索と同じ答えになるとは限りません。

In [ ]:
# TODO 2：初期配分を変更する / TODO 3：目的関数を変更する
P2 = em.with_changes(PARAMS, returns_exponent=0.8)       # 例：収穫逓減を弱める
OBJ2 = "final_living_standard"
starts = {"near equal": None,
          "education 100%": {"education": 1.0, "rd": 0.0, "infrastructure": 0.0, "healthcare": 0.0},
          "healthcare 100%": {"education": 0.0, "rd": 0.0, "infrastructure": 0.0, "healthcare": 1.0}}
for name, start in starts.items():
    r = op.compare_algorithms(P2, OBJ2, STEP, start=start)
    print(f"初期配分 {name:16}: " + " | ".join(
        f"{x['algorithm']} {x['objective']:.4f} (gap {x['gap']['absolute']:.4f}) {al.format_shares(x['shares'])}"
        for x in r.values()))
# TODO 4：結果を比較する（なぜ差が出るのか、どの条件で出るのか）

## 考察（チームで記入）

- **TODO**：3 つのアルゴリズムの目的関数値・評価回数・実行時間を比べて、何が言えるか（RQ4）
- **TODO**：Greedy 法・局所探索が全探索に届かなかった条件はあるか。その理由は何か
- **TODO**：刻みを細かくすると、候補数と実行時間はどう増えたか。理論計算量と合っているか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）